In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_11_thermostats_and_barostats.yaml')
ws.start()


$\def\AA{\text{Å}}$

# Lesson 11 — Thermostats, Barostats & Extended-Lagrangian Ensembles

_Session 29039f5e · Estimated time: ~120 minutes · 41 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Distinguish the four primary statistical mechanical ensembles used in atomistic simulation—Microcanonical (NVE), Canonical (NVT), Isobaric-Isothermal (NPT), and Grand Canonical (μVT)—and identify their fixed control variables and minimized thermodynamic potentials.
- Explain why starting an MD trajectory from an ordered crystal lattice or dilute gas causes spontaneous kinetic temperature drift in NVE, and why ad-hoc velocity rescaling (Quench/Berendsen) fails to generate the true canonical ensemble.
- Analyze the Andersen stochastic thermostat (Poisson heat-bath collisions at rate ν), including its rigorous canonical configurational sampling and its degradation of momentum conservation, velocity autocorrelation tails, and self-diffusivity D(ν).
- Derive and apply the Nosé–Hoover deterministic extended-Lagrangian thermostat, its feedback friction variable ζ(t), thermal inertia Q, conserved extended energy H', Martyna–Tuckerman–Tobias–Klein (MTTK) Trotter operator-splitting propagator, and why Nosé–Hoover Chains are required to cure non-ergodicity in harmonic systems.
- Compare constant-volume (NVT) and constant-pressure (NPT) phase diagrams, and formulate the Andersen isotropic barostat and Parrinello–Rahman variable-shape (3×3 cell matrix L) barostat for simulating solid-solid martensitic phase transformations such as KCl rocksalt (B1) to CsCl (B2).


In [2]:
ws.render_section('sec_ensembles_and_quench')

## 1. Thermodynamic Ensembles, Temperature Drift, and Velocity Rescaling

Standard Newton's equations of motion ($\dot{\mathbf{r}}_i = \mathbf{p}_i/m_i$, $\dot{\mathbf{p}}_i = \mathbf{F}_i$) conserve particle number $N$, box volume $V$, and total mechanical energy $E = K + V$. Thus, unperturbed Molecular Dynamics naturally samples the **microcanonical ($NVE$) ensemble**, whose fundamental thermodynamic potential is the entropy $S(N, V, E)$.

Laboratory experiments, however, are rarely isolated at constant energy $E$ and fixed volume $V$. Instead, macroscopic measurements are typically performed at controlled temperature $T$, pressure $P$, or chemical potential $\mu$:

| Ensemble | Fixed Variables | Conjugates Allowed to Fluctuate | Minimized Thermodynamic Potential |
| :--- | :---: | :---: | :--- |
| **Microcanonical** | $(N, V, E)$ | $T, P, \mu$ | $-T S(N,V,E)$ (maximizes $S$) |
| **Canonical** | $(N, V, T)$ | $E$ (kinetic $K$ & potential $V$), $P$ | Helmholtz Free Energy $F = E - TS$ |
| **Isobaric-Isothermal** | $(N, P, T)$ | $E$, Volume $V$ (and box shape $\mathbf{L}$) | Gibbs Free Energy $G = E + PV - TS$ |
| **Grand Canonical** | $(\mu, V, T)$ | $E$, Particle Number $N$ | Grand Potential $\Phi_G = F - \mu N = -PV$ |

### Initializing Velocities and Spontaneous Temperature Drift
To start an MD simulation at a desired temperature $T$, we sample initial particle velocities from the **Maxwell–Boltzmann (Gaussian) distribution**:
$$P(v_\alpha)\,dv_\alpha = \left(\frac{m}{2\pi k_B T}\right)^{1/2} \exp\!\left(-\frac{m v_\alpha^2}{2 k_B T}\right) dv_\alpha$$
and shift the velocities so that the total center-of-mass linear momentum vanishes ($\sum_i m_i \mathbf{v}_i = \mathbf{0}$), leaving $g = 3N - 3$ translational degrees of freedom. The instantaneous kinetic temperature $T_{\text{kin}}$ (often written $T_I$) is defined via equipartition:
$$k_B T_I = \frac{1}{3N - 3} \sum_{i=1}^N m_i v_i^2 = \frac{2 K}{3N - 3}$$

Even if $T_I(0) = T_0$ initially, a microcanonical ($NVE$) run will **not** stay at $T_0$:
- **Starting from a perfect crystal lattice** ($V$ near its potential minimum): As atoms move off their ideal lattice sites and the structure becomes disordered (or melts), potential energy $V$ increases. Because $E = K + V$ is conserved, the system **sucks up kinetic energy and cools down** ($T_I$ drops).
- **Starting from an unrelaxed high-energy configuration or dilute gas** that condenses into attractive potential wells: Potential energy $V$ decreases and converts into kinetic energy, causing the system to **heat up**.

### The Quench (Velocity Rescaling) and Berendsen Methods
The simplest way to force the system toward target temperature $T$ is the **Quench (velocity rescaling) method**: run MD for a slice of steps, compute $T_I$, and rescale every particle's velocity by
$$\mathbf{v}_i^{\text{new}} = \mathbf{v}_i^{\text{old}} \sqrt{\frac{T}{T_I}}$$
Wait until $T_I$ drifts again, repeat a few quenches during equilibration, and then turn quenching off (`QUENCH 0`) for production.

A gentler continuous variant is the **Berendsen weak-coupling thermostat**, which rescales velocities at every step $\Delta t$ with relaxation time $\tau_T$:
$$\lambda_B = \sqrt{1 + \frac{\Delta t}{\tau_T}\left(\frac{T}{T_I} - 1\right)}$$

> [!WARNING] Why Velocity Rescaling Is Not a Canonical ($NVT$) Thermostat
> In a true canonical ($NVT$) ensemble, kinetic energy $K$ is **not** constant; it fluctuates with variance
> $$\langle \Delta K^2 \rangle_{NVT} = \frac{g}{2}(k_B T)^2 \implies \frac{\sigma_{T_I}}{T} = \sqrt{\frac{2}{g}} \approx \sqrt{\frac{2}{3N}}$$
> Rigidly clamping $T_I = T$ (or damping fluctuations via Berendsen coupling) suppresses canonical energy fluctuations to $\mathcal{O}(1/N)$ errors, violates time-reversibility and Liouville phase-space measure conservation, and can trigger the notorious **"flying ice cube" artifact**—where kinetic energy drains out of high-frequency internal vibrational modes into collective center-of-mass translation or rotation, freezing the molecule internally while it hurtles through space!


In [3]:
ws.render_section('q01_ensemble_potentials')

**Q1 · easy** · 💡

In an atomistic simulation where the particle number $N$, external pressure $P$, and temperature $T$ are held fixed while the system energy $E$ and simulation box volume $V$ fluctuate in equilibrium, which statistical mechanical ensemble is being sampled and which thermodynamic potential reaches a minimum at equilibrium?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall which thermodynamic potential transforms $(S, V) \to (T, P)$ via Legendre transformation of internal energy $E$.

</div>
</details>

- [ ] **(A)** Microcanonical ($NVE$) ensemble; entropy $S$ is minimized.
- [ ] **(B)** Canonical ($NVT$) ensemble; Helmholtz free energy $F = E - TS$ is minimized.
- [ ] **(C)** Isobaric-isothermal ($NPT$) ensemble; Gibbs free energy $G = E + PV - TS$ is minimized.
- [ ] **(D)** Grand canonical ($\mu VT$) ensemble; grand potential $\Phi_G = F - \mu N$ is minimized.

In [4]:
ws.render_section('q02_grand_canonical_variables')

**Q2 · easy** · 💡

Which set of macroscopic variables is held fixed in the **Grand Canonical ensemble**, making it the natural choice for simulating open adsorption problems (such as gas uptake in a nanoporous zeolite at fixed external reservoir pressure)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In an open system that can exchange both heat and particles with an infinite reservoir, what intensive variable controls particle exchange?

</div>
</details>

- [ ] **(A)** Particle number $N$, volume $V$, and total energy $E$.
- [ ] **(B)** Chemical potential $\mu$, volume $V$, and temperature $T$.
- [ ] **(C)** Chemical potential $\mu$, pressure $P$, and entropy $S$.
- [ ] **(D)** Particle number $N$, pressure $P$, and enthalpy $H$.

In [5]:
ws.render_section('q03_lattice_start_cooling')

**Q3 · easy** · 💡

You initialize a microcanonical ($NVE$) Molecular Dynamics simulation of $N = 256$ Argon atoms on a **perfect zero-strain FCC crystal lattice** ($V(0)$ at its minimum) and draw initial velocities from a Maxwell–Boltzmann distribution at $T_0 = 120\text{ K}$. Over the first picosecond (without any thermostat), what happens to the instantaneous kinetic temperature $T_I(t)$ and why?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In an NVE simulation, $E = K + V$ is strictly conserved. What happens to the potential energy $V$ as atoms move away from their minimum-energy ideal lattice positions?

</div>
</details>

- [ ] **(A)** $T_I(t)$ remains pinned at $120\text{ K}$ because the initial velocities were drawn from an exact Maxwell–Boltzmann distribution.
- [ ] **(B)** $T_I(t)$ drops to roughly half its initial value (the system cools down) because disordering the perfect lattice increases potential energy $V$, which drains kinetic energy $K$ under constant $E = K + V$.
- [ ] **(C)** $T_I(t)$ doubles (the system heats up) because phonon modes amplify the initial velocities.
- [ ] **(D)** $T_I(t)$ drops to $0\text{ K}$ because total linear momentum conservation cancels opposite velocities.

In [6]:
ws.render_section('q04_velocity_rescale_numeric')

**Q4 · easy** · 💡

During equilibration of an Argon liquid toward target reduced temperature $T = 1.35$, you measure an instantaneous kinetic temperature of $T_I = 0.60$. In a velocity-rescaling (**Quench**) step $\mathbf{v}_i^{\text{new}} = \lambda \mathbf{v}_i^{\text{old}}$, what numerical scaling factor $\lambda$ must multiply every particle's velocity?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\lambda = \sqrt{T / T_I} = \sqrt{1.35 / 0.60}$.

</div>
</details>

_Numeric answer required._

In [7]:
ws.render_section('q05_instantaneous_temp_dof')

**Q5 · medium** · 💡

Consider a 3D periodic MD simulation of $N = 101$ identical atoms with zero total center-of-mass momentum ($\mathbf{P}_{\text{total}} = \mathbf{0}$). At a given time step, the total kinetic energy is $K = \frac{1}{2}\sum_{i=1}^{N} m v_i^2 = 450.0$ (in reduced units where $k_B = 1$). Using the exact degree-of-freedom formula $k_B T_I = \frac{2K}{3N - 3}$, compute the instantaneous kinetic temperature $T_I$.


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

With $N = 101$, the number of independent translational degrees of freedom is $g = 3N - 3 = 3(101) - 3 = 300$. Then $T_I = 2K / g$.

</div>
</details>

_Numeric answer required._

In [8]:
ws.render_section('q06_canonical_temp_fluctuation')

**Q6 · medium** · 💡

In a true canonical ($NVT$) ensemble with $g = 3N = 600$ degrees of freedom at target temperature $T = 300\text{ K}$, what is the theoretical standard deviation $\sigma_{T_I} = T \sqrt{2/g}$ of the instantaneous kinetic temperature fluctuations (in $\text{K}$, to two decimal places)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $\sigma_{T_I} = 300 \times \sqrt{2 / 600} = 300 / \sqrt{300} = 10\sqrt{3}$.

</div>
</details>

_Numeric answer required._

In [9]:
ws.render_section('q07_berendsen_defects')

**Q7 · medium** · 💡

Why are **continuous velocity rescaling (Quench)** and the **Berendsen weak-coupling thermostat** generally avoided for production equilibrium and dynamical measurements in modern MD? *(Select all that apply.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider canonical kinetic energy fluctuations, time-reversibility, and the 'flying ice cube' equipartition violation.

</div>
</details>

- [ ] **(A)** They suppress natural canonical kinetic energy fluctuations ($\langle \Delta K^2 \rangle$), failing to sample the true canonical ($NVT$) phase-space distribution.
- [ ] **(B)** They can induce the 'flying ice cube' effect, draining kinetic energy from high-frequency internal vibrations into low-frequency translational/rotational motion.
- [ ] **(C)** They violate time-reversibility and do not possess a conserved extended Hamiltonian.
- [ ] **(D)** They require inverting a $3N \times 3N$ matrix at every Verlet step, making them $\mathcal{O}(N^3)$.

In [10]:
ws.render_section('sec_andersen_thermostat')

## 2. The Andersen Stochastic Thermostat & Brownian Collision Dynamics

To rigorously generate the **canonical ($NVT$) distribution**, Hans C. Andersen (1980) proposed coupling the MD system to an imaginary **stochastic heat bath** at temperature $T$ via random, instantaneous collisions—physically analogous to a macromolecule or colloid buffeted by surrounding solvent molecules, or a crystal lattice coupled to a bath of phonons/photons.

### Algorithm & Integration with Velocity Verlet
Each particle experiences independent **Poisson collisions** with the heat bath at average collision rate $\nu$ (having units of $\text{time}^{-1}$):
1. Advance positions and velocities over one step $\Delta t$ using standard **Velocity Verlet** (updating $\mathbf{v}(t+\Delta t/2)$, $\mathbf{r}(t+\Delta t)$, Forces $\mathbf{F}(t+\Delta t)$, and $\mathbf{v}(t+\Delta t)$).
2. After the second velocity half-step, loop over each particle $i = 1, \dots, N$:
   - Draw a uniform random number $u_i \sim \text{Uniform}(0, 1)$.
   - If $u_i < \nu \Delta t$ (the first-order Poisson collision probability in $\Delta t$; exactly $1 - e^{-\nu\Delta t}$), **discard** $\mathbf{v}_i$ and draw all three components $(v_{ix}, v_{iy}, v_{iz})$ fresh from the **Maxwell–Boltzmann (Gaussian) distribution** at target temperature $T$:
     $$v_{i\alpha}^{\text{new}} \sim \mathcal{N}\!\left(0,\, \sigma_v^2 = \frac{k_B T}{m_i}\right)$$

### Advantages vs. Dynamical Pitfalls of Andersen Thermostatting
- **Rigorous Static $(T, V, N)$ Sampling:** Because deterministic Verlet propagation preserves phase-space volume (Liouville's theorem) and remains on constant-energy surfaces while stochastic Maxwellian kicks jump between energy shells according to detailed balance, the stationary probability distribution is **guaranteed** to be the exact canonical Boltzmann distribution $P(\mathbf{r}, \mathbf{p}) \propto \exp[-\beta (K + V)]$ for **any** positive collision frequency $\nu > 0$.
- **Discontinuous Velocities & Broken Conservation Laws:** Each random velocity reassignment introduces a kink in the trajectory $\mathbf{r}_i(t)$ (discontinuous $\mathbf{v}_i$) and **destroys conservation of total linear momentum** $\mathbf{P} = \sum m_i \mathbf{v}_i$ as well as energy.
- **Destruction of Velocity Correlations & Diffusivity Suppression:** Between collisions, the survival probability that a particle has *not* been randomized after elapsed time $t$ is $e^{-\nu t}$. Consequently, the velocity autocorrelation function (VACF) is exponentially damped:
  $$C_{vv}^{(\nu)}(t) = \langle \mathbf{v}_i(0)\cdot\mathbf{v}_i(t) \rangle_\nu \approx C_{vv}^{(0)}(t)\, e^{-\nu t}$$
  By the **Green–Kubo formula**, the self-diffusion coefficient $D = \frac{1}{3}\int_0^\infty C_{vv}(t)\,dt$ (and equivalently the long-time slope of the mean-squared displacement $\langle \Delta r^2(t) \rangle = 6 D t$, as shown in Frenkel & Smit Fig. 6.3) is artificially **suppressed** as $\nu$ increases:
  $$D(\nu) \approx \frac{k_B T}{m(\gamma_0 + \nu)} = \frac{D_0}{1 + \nu \tau_0}$$
  When $\nu$ is very large ($\nu \to \infty$), particles have their velocities reversed and randomized before they can move a fraction of an interatomic spacing, trapping them in place ($D \to 0$).


In [11]:
ws.render_section('concept_andersen_vacf')

### Interactive Explorer: Andersen Collision Rate ν, VACF Decorrelation, and Self-Diffusion D(ν)

Adjust the **Andersen collision rate $\nu$** to observe how stochastic heat-bath kicks multiply the intrinsic Newtonian velocity autocorrelation function $C_{vv}(t)$ by $e^{-\nu t}$, shrinking the Green–Kubo integral area (shaded) and flattening the mean-squared displacement $\langle \Delta r^2(t) \rangle$ in the inset (reproducing Frenkel & Smit Figure 6.3 from the lecture slides).


**Key Takeaway:** Static canonical averages (such as $\langle V \rangle$, pressure $P$, and radial distribution function $g(r)$) are completely independent of $\nu > 0$. However, **dynamical transport properties** ($D$, viscosity $\eta$, thermal conductivity $\kappa$, and long-time $t^{-3/2}$ hydrodynamic tails) are corrupted whenever $\nu$ is comparable to or larger than the intrinsic collision rate $1/\tau_c$.


In [12]:
ws.render_section('q08_andersen_collision_prob')

**Q8 · easy** · 💡

An Andersen thermostat operates with Poisson collision frequency $\nu = 4.0\text{ ps}^{-1}$ and MD integration time step $\Delta t = 0.005\text{ ps}$. Using the linear first-order rate approximation $P_{\text{coll}} = \nu \Delta t$, what is the probability that a given atom undergoes a stochastic heat-bath collision during a single time step?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Multiply the collision rate $\nu$ by the time step $\Delta t$.

</div>
</details>

_Numeric answer required._

In [13]:
ws.render_section('q09_andersen_expected_kicks')

**Q9 · easy** · 💡

In an MD system of $N = 500$ atoms thermostatted by an Andersen bath with collision probability $\nu \Delta t = 0.015$ per atom per step, how many total velocity reassignments occur on average across the entire system during $100$ integration steps?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Expected collisions $= N \times (\nu \Delta t) \times N_{\text{steps}}$.

</div>
</details>

_Numeric answer required._

In [14]:
ws.render_section('q10_andersen_verlet_placement')

**Q10 · easy** · 💡

When incorporating the **Andersen thermostat** into the standard **Velocity Verlet** integrator, where in the time-step cycle is the stochastic Maxwell–Boltzmann velocity resampling step performed?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check slide 5 of the lecture notes: Velocity Verlet has two half-step velocity updates and one full-step position update.

</div>
</details>

- [ ] **(A)** Immediately after the second velocity half-step update $\mathbf{v}(t+\Delta t) = \mathbf{v}(t+\Delta t/2) + \frac{1}{2}\mathbf{a}(t+\Delta t)\Delta t$, at the end of the full step.
- [ ] **(B)** Between the position update $\mathbf{r}(t+\Delta t)$ and the force evaluation $\mathbf{F}(t+\Delta t)$, while velocities are at the half-step $t+\Delta t/2$.
- [ ] **(C)** By adding a random Gaussian displacement directly to the atomic coordinates $\mathbf{r}_i(t)$ before computing forces.
- [ ] **(D)** By rescaling all atomic forces $\mathbf{F}_i$ by a Poisson random integer.

In [15]:
ws.render_section('q11_andersen_msd_ordering')

**Q11 · medium** · 💡

In Frenkel & Smit Figure 6.3 (shown on Slide 5 and in the interactive explorer above), the mean-squared displacement $\langle \Delta r(t)^2 \rangle$ of a Lennard-Jones fluid ($T = 2.0, N = 108$) is plotted versus time $t$ for four Andersen collision frequencies: $\nu = 0, 5, 10, 20$. Which curve exhibits the **steepest** long-time linear slope $6 D(\nu)$, and which exhibits the **shallowest** slope?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Does increasing the stochastic collision frequency $\nu$ speed up or slow down long-range particle diffusion?

</div>
</details>

- [ ] **(A)** Steepest slope at $\nu = 20$; shallowest slope at $\nu = 0$.
- [ ] **(B)** Steepest slope at $\nu = 0$ (unperturbed Newtonian dynamics); shallowest slope at $\nu = 20$ (frequent randomizing collisions).
- [ ] **(C)** All four curves have the exact same long-time slope because $D$ is an equilibrium canonical average.
- [ ] **(D)** Steepest slope at intermediate $\nu = 10$ due to stochastic resonance; $\nu = 0$ has zero slope.

In [16]:
ws.render_section('q12_andersen_static_vs_dynamic')

**Q12 · medium** · 💡

Suppose you run two long, well-equilibrated $NVT$ simulations of liquid Argon at $T = 1.5$ and $\rho = 0.8$: Simulation A uses an Andersen thermostat with $\nu = 0.5$, and Simulation B uses $\nu = 15.0$. How will their **radial distribution functions $g(r)$** and **self-diffusion coefficients $D$** compare?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Distinguish between static configurational averages $\langle A(\mathbf{r}) \rangle_{NVT}$ and time-dependent correlation functions $\langle \mathbf{v}(0)\cdot\mathbf{v}(t) \rangle$.

</div>
</details>

- [ ] **(A)** Both $g(r)$ and $D$ will be identical between Simulation A and Simulation B.
- [ ] **(B)** Their static radial distribution functions $g(r)$ will be identical within statistical noise, but Simulation B ($\nu = 15.0$) will have a significantly smaller self-diffusion coefficient $D$.
- [ ] **(C)** Simulation B will have sharper crystalline peaks in $g(r)$ because high $\nu$ freezes the configurational distribution.
- [ ] **(D)** Simulation B will have a larger self-diffusion coefficient $D$ because frequent kicks inject extra kinetic energy.

In [17]:
ws.render_section('q13_andersen_diffusivity_calc')

**Q13 · medium** · 💡

Suppose the intrinsic velocity autocorrelation function of a simple fluid at $\nu = 0$ is purely exponential, $C_{vv}^{(0)}(t) = \langle v^2 \rangle e^{-t/\tau_0}$ with relaxation rate $\gamma_0 = 1/\tau_0 = 2.0\text{ ps}^{-1}$, giving true self-diffusion coefficient $D(0) = \frac{1}{3}\int_0^\infty C_{vv}^{(0)}(t)\,dt = 1.20 \times 10^{-4}\text{ cm}^2/\text{s}$. If an Andersen thermostat is turned on with collision frequency $\nu = 6.0\text{ ps}^{-1}$ so that $C_{vv}^{(\nu)}(t) = C_{vv}^{(0)}(t)\,e^{-\nu t}$, what is the measured self-diffusion coefficient $D(\nu)$ (in units of $10^{-4}\text{ cm}^2/\text{s}$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Notice that $\int_0^\infty e^{-(\gamma_0 + \nu)t}\,dt = 1/(\gamma_0 + \nu)$, so $D(\nu) = D(0) \frac{\gamma_0}{\gamma_0 + \nu}$.

</div>
</details>

_Numeric answer required._

In [18]:
ws.render_section('q14_andersen_properties_multi')

**Q14 · medium** · 💡

Which of the following statements accurately describe the **Andersen thermostat**? *(Select all that apply.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check conservation of momentum, continuity of velocities, Boltzmann sampling, and physical interpretation.

</div>
</details>

- [ ] **(A)** It produces trajectories with discontinuous velocities whenever a bath collision occurs.
- [ ] **(B)** It strictly conserves total linear momentum $\mathbf{P} = \sum_i m_i \mathbf{v}_i$ at every step.
- [ ] **(C)** It is guaranteed to sample configurations according to the canonical Boltzmann probability $\propto e^{-\beta H}$.
- [ ] **(D)** It is physically well-suited for modeling a large solute (such as a polymer or colloid) immersed in an implicit solvent bath.

In [19]:
ws.render_section('sec_nose_hoover_theory')

## 3. The Nosé–Hoover Deterministic Thermostat & Extended Lagrangians

Can we sample the canonical ($NVT$) ensemble **deterministically** and **time-reversibly**, without random numbers or velocity discontinuities? In 1984, Shuichi Nosé introduced the **extended-Lagrangian** method, which William G. Hoover (1985) cast into a constant-time-step form now universally known as the **Nosé–Hoover thermostat**.

### Extended Phase Space and Integral Feedback Friction $\zeta(t)$
We augment the physical $N$-particle system with an artificial 1D heat-bath degree of freedom characterized by logarithmic coordinate $\eta = \ln s$, conjugate velocity/friction variable $\zeta = \frac{d\ln s}{dt} = \dot{\eta}$, and **fictitious thermal inertia ("heat bath mass") $Q$**:
$$\frac{d\mathbf{r}_i}{dt} = \frac{\mathbf{p}_i}{m_i}, \qquad \frac{d\mathbf{p}_i}{dt} = \mathbf{F}_i - \zeta(t)\,\mathbf{p}_i$$
$$Q\frac{d\zeta}{dt} = \sum_{i=1}^N \frac{p_i^2}{2m_i} - \frac{g}{2}k_B T \quad \Longleftrightarrow \quad \frac{d\zeta}{dt} = \frac{1}{Q}\left(\sum_{i=1}^N \frac{p_i^2}{m_i} - g k_B T\right)$$
where $g = 3N$ (or $3N-3$) is the number of degrees of freedom (note that in Hoover's standard notation used on Slide 7, the factor of $2$ is absorbed into $Q$ so $\dot{\zeta} = \frac{1}{Q}\big(\sum \frac{p_i^2}{m_i} - 3Nk_B T\big)$).

Look closely at how the dynamical friction coefficient $\zeta(t)$ acts as an **integral feedback controller**:
- **System too hot ($2K > g k_B T$):** $\dot{\zeta} > 0$, so $\zeta$ increases and becomes positive ($+\zeta \mathbf{p}_i$ acts as viscous drag, removing kinetic energy).
- **System too cold ($2K < g k_B T$):** $\dot{\zeta} < 0$, so $\zeta$ decreases and becomes **negative** ($-\zeta \mathbf{p}_i$ acts as "anti-friction" or negative damping, pumping kinetic energy from the reservoir into the particles!).
- **At steady state ($\langle d\zeta/dt \rangle = 0$):** Time-averaging $\dot{\zeta}$ immediately forces $\left\langle \sum \frac{p_i^2}{2m_i} \right\rangle = \frac{g}{2}k_B T$, guaranteeing exact canonical equipartition!

### Conserved Extended Energy $H'$ and Stationary Measure
Although the physical system energy $H = K + V$ fluctuates as heat flows in and out of the reservoir, the **extended energy of system + heat bath** is a strict constant of motion:
$$H' = K(\mathbf{p}) + V(\mathbf{r}) + \frac{Q}{2}\zeta^2 + g k_B T \,\eta, \qquad \text{where } \frac{d\eta}{dt} = \frac{d\ln s}{dt} = \zeta$$
Checking $\frac{dH'}{dt} = 0$ in an MD code is the gold-standard diagnostic for verifying that your Nosé–Hoover integrator and time step $\Delta t$ are bug-free! Provided the extended trajectory $(\mathbf{r}, \mathbf{p}, \zeta)$ is **ergodic** on the constant-$H'$ hypersurface, projecting out $(\zeta, \eta)$ yields the exact canonical stationary distribution:
$$\rho_{\text{stat}}(\mathbf{r}, \mathbf{p}, \zeta) \propto \exp\!\left[-\beta\left(V(\mathbf{r}) + \sum_{i=1}^N \frac{p_i^2}{2m_i} + \frac{Q}{2}\zeta^2\right)\right]$$

### Choosing the Thermal Inertia ("Bath Mass") $Q$
As illustrated on Slide 8 ($Q = 1$ vs. $Q = 100$ for 256 Argon atoms):
- **Small $Q$ ($Q \to 0$, strong coupling):** $\zeta$ reacts violently to tiny kinetic energy deviations, producing **fast, high-frequency temperature oscillations** ($\omega_{\text{NH}} \propto \sqrt{g k_B T / Q}$) that can resonate unphysically with bond vibrations and require smaller $\Delta t$.
- **Large $Q$ ($Q \to \infty$, weak coupling):** $\dot{\zeta} \to 0$, recovering standard microcanonical ($NVE$) dynamics if $\zeta(0)=0$, but resulting in **very slow thermal equilibration** over short runs.


In [20]:
ws.render_section('concept_nh_thermal_inertia')

### Interactive Explorer: Nosé–Hoover Thermal Inertia Q and Temperature Relaxation

Compare how the **thermal inertia $Q$** ("fictitious heat-bath mass" from Slide 8) governs the oscillation frequency and damping timescale of the instantaneous kinetic temperature $T_{\text{kin}}(t)$ after a temperature jump from $T_{\text{init}}$ to $T_0$.


**Key Takeaway:** Neither $Q \to 0$ nor $Q \to \infty$ is desirable. In practice, $Q$ is chosen so that the thermostat characteristic period $\tau_{\text{NH}} = 2\pi\sqrt{Q/(2 g k_B T_0)}$ matches typical vibrational/structural relaxation timescales of the material ($\tau_{\text{NH}} \sim 20\text{–}100\,\Delta t$).


In [21]:
ws.render_section('q15_nh_feedback_sign')

**Q15 · easy** · 💡

In the Nosé–Hoover equations of motion, $\dot{\mathbf{p}}_i = \mathbf{F}_i - \zeta \mathbf{p}_i$ and $Q\dot{\zeta} = \sum_i \frac{p_i^2}{m_i} - 3Nk_B T$. Suppose at time $t$, the instantaneous kinetic energy is **lower** than the target thermal energy ($\sum_i \frac{p_i^2}{m_i} < 3Nk_B T$) and $\zeta(t)$ becomes **negative** ($\zeta < 0$). What physical effect does the term $-\zeta \mathbf{p}_i$ have on the particles?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

If $\zeta < 0$, what is the sign of $-\zeta$ multiplying the momentum $\mathbf{p}_i$?

</div>
</details>

- [ ] **(A)** It acts as a viscous drag force that slows the particles down further.
- [ ] **(B)** It acts as 'negative friction' (anti-damping) parallel to $\mathbf{p}_i$, accelerating the particles and pumping kinetic energy from the heat reservoir into the system.
- [ ] **(C)** It rotates the particle momenta perpendicular to their velocities without changing kinetic energy.
- [ ] **(D)** It randomly reassigns velocities from a Gaussian distribution.

In [22]:
ws.render_section('q16_nh_zeta_rate_numeric')

**Q16 · medium** · 💡

Using the Nosé–Hoover friction equation on Slide 7,
$$\frac{d\zeta}{dt} = \frac{1}{Q}\left(\sum_{i=1}^N \frac{p_i^2}{m_i} - 3N k_B T\right)$$
consider a system with $3N = 300$ degrees of freedom at target temperature $k_B T = 1.20$ and bath mass $Q = 40.0$. At an instant when the total kinetic energy is $K = \frac{1}{2}\sum_{i=1}^N \frac{p_i^2}{m_i} = 220.0$, what is the instantaneous time derivative $\frac{d\zeta}{dt}$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

First note that $\sum \frac{p_i^2}{m_i} = 2K = 440.0$, while $3N k_B T = 300 \times 1.20 = 360.0$. Then divide their difference by $Q = 40.0$.

</div>
</details>

_Numeric answer required._

In [23]:
ws.render_section('q17_nh_conserved_hamiltonian')

**Q17 · medium** · 💡

In a Nosé–Hoover canonical ($NVT$) simulation, the physical system energy $H = K(\mathbf{p}) + V(\mathbf{r})$ fluctuates in time. Which quantity $H'$ is **strictly conserved** along the exact trajectory (and serves as the primary numerical check for integrator accuracy)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at Slide 7: $H'$ adds the kinetic energy of the friction variable $\zeta$ and the potential energy $3N k_B T \ln(s)$ of the reservoir coordinate.

</div>
</details>

- [ ] **(A)** $H' = K + V - \zeta^2$
- [ ] **(B)** $H' = K + V + \frac{Q}{2}\zeta^2 + 3N k_B T \ln(s)$, where $\frac{d\ln s}{dt} = \zeta$
- [ ] **(C)** $H' = K - V + Q\zeta$
- [ ] **(D)** $H' = (K + V)\exp(-\zeta t)$

In [24]:
ws.render_section('q18_nh_large_Q_limit')

**Q18 · easy** · 💡

What happens to a Nosé–Hoover simulation initialized with $\zeta(0) = 0$ in the limit where the fictitious heat-bath mass becomes infinitely large ($Q \to \infty$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Inspect $\dot{\zeta} = \frac{1}{Q}(\sum p_i^2/m_i - 3Nk_B T)$ as $Q \to \infty$.

</div>
</details>

- [ ] **(A)** The system decouples from the heat bath ($\dot{\zeta} \to 0$, so $\zeta(t) = 0$) and reduces to standard microcanonical ($NVE$) Molecular Dynamics.
- [ ] **(B)** The velocities are rescaled instantaneously at every time step (Quench limit).
- [ ] **(C)** The friction coefficient $\zeta$ oscillates at infinite frequency, blowing up the integrator.
- [ ] **(D)** The system freezes immediately to $T = 0\text{ K}$.

In [25]:
ws.render_section('q19_nh_Q1_vs_Q100_traces')

**Q19 · medium** · 💡

On Slide 8 ("Effect of thermostat"), two temperature traces of $N = 256$ Argon atoms are compared: one with `NOSE 1.` ($Q = 1$) and one with `NOSE 100.` ($Q = 100$). Why does the $Q = 1$ trace display a pronounced, regular sinusoidal ringing with period $\approx 3\text{–}4$ time units after the initial temperature spike, whereas $Q = 100$ fluctuates much more gently?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall how the natural oscillation frequency $\omega_{\text{NH}}$ of the coupled $(K, \zeta)$ feedback loop depends on $Q$.

</div>
</details>

- [ ] **(A)** Because $Q = 1$ has very small thermal inertia, causing $\zeta(t)$ to over-react rapidly to kinetic energy deviations with oscillation frequency $\omega_{\text{NH}} \propto Q^{-1/2}$.
- [ ] **(B)** Because $Q = 1$ turns on a stochastic Poisson random number generator every step.
- [ ] **(C)** Because $Q = 100$ disables periodic boundary conditions.
- [ ] **(D)** Because $Q = 1$ changes the Lennard-Jones potential cutoff from $2.5\sigma$ to $1.0\sigma$.

In [26]:
ws.render_section('q20_nh_zeta_variance_numeric')

**Q20 · medium** · 💡

According to the stationary Nosé–Hoover phase-space distribution on Slide 7,
$$\rho(\mathbf{r}, \mathbf{p}, \zeta) \propto \exp\!\left[-\beta\left(V(\mathbf{r}) + \sum_i \frac{p_i^2}{2m_i} + \frac{Q}{2}\zeta^2\right)\right]$$
the friction variable $\zeta$ at equilibrium is an independent Gaussian random variable with mean $\langle \zeta \rangle = 0$. If $k_B T = 1/\beta = 2.40$ and $Q = 6.0$, what is the equilibrium variance $\langle \zeta^2 \rangle$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare $\exp(-\beta \frac{Q}{2}\zeta^2)$ with a standard Gaussian $\exp(-\zeta^2 / (2\sigma_\zeta^2))$ to find $\sigma_\zeta^2 = \frac{k_B T}{Q}$.

</div>
</details>

_Numeric answer required._

In [27]:
ws.render_section('q21_nose_vs_hoover_historical')

**Q21 · easy** · 💡

As noted on Slide 7, what was the key practical modification that **Hoover (1985)** made to **Nosé's original (1984)** extended-Lagrangian formulation?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In Nosé's original Lagrangian, the variable $s$ scaled real time $dt = s\,dt'$. Why was that awkward in MD?

</div>
</details>

- [ ] **(A)** Nosé's original algorithm required unevenly spaced, fluctuating physical time steps ($dt \propto s$), whereas Hoover reformulated the equations in real time with a constant time step $\Delta t$ and dynamical friction $\zeta = \dot{s}/s$.
- [ ] **(B)** Hoover replaced Nosé's deterministic differential equations with Poisson random collisions.
- [ ] **(C)** Hoover restricted the thermostat to 1D ideal gases.
- [ ] **(D)** Hoover eliminated the thermal inertia parameter $Q$ so no user parameter is needed.

In [28]:
ws.render_section('sec_nh_propagation_and_chains')

### Martyna–Tuckerman–Tobias–Klein (MTTK) Trotter Propagation
Because the Nosé–Hoover momentum equation $\dot{\mathbf{p}} = \mathbf{F} - \zeta\mathbf{p}$ contains a **velocity-dependent force** and is not canonical Hamiltonian in $(\mathbf{r}, \mathbf{p})$ alone, naive Euler or Verlet updates lose time-reversibility and cause drift in $H'$.

Martyna et al. (1996) solved this by decomposing the total Liouville time-evolution operator $e^{iL\delta t}$, where $iL = iL_1 + iL_2 + iL_3 + iL_4$:
$$\underbrace{iL_1 = \mathbf{v}\cdot\frac{\partial}{\partial \mathbf{r}}, \qquad iL_2 = \mathbf{f}\cdot\frac{\partial}{\partial \mathbf{p}}}_{\text{System Hamiltonian (Velocity Verlet)}} \qquad\qquad \underbrace{iL_3 = \zeta\frac{\partial}{\partial \eta} - \zeta\mathbf{p}\cdot\frac{\partial}{\partial \mathbf{p}}, \qquad iL_4 = \left(\frac{p^2}{m} - 3Nk_B T\right)Q^{-1}\frac{\partial}{\partial \zeta}}_{\text{Thermal Reservoir + Friction Interaction}}$$

Each individual sub-operator can be integrated **analytically**:
- $\exp(iL_1\delta t)\mathbf{r} = \mathbf{r} + \mathbf{v}\delta t$ *(position drift)*
- $\exp(iL_2\delta t)\mathbf{p} = \mathbf{p} + \mathbf{f}\delta t$ *(force kick)*
- $\exp(iL_3\delta t)\eta = \eta + \zeta\delta t$ and $\exp(iL_3\delta t)\mathbf{p} = \mathbf{p}\exp(-\zeta\delta t)$ *(exact exponential friction scaling!)*
- $\exp(iL_4\delta t)\zeta = \zeta + \left(\frac{p^2}{m} - 3Nk_B T\right)Q^{-1}\delta t$ *(bath velocity kick)*

Using the **symmetric Trotter factorization**, we sandwich the Velocity Verlet step $\exp(i(L_1+L_2)\delta t)$ between two half-steps of the reservoir update $\exp(i(L_3+L_4)\delta t/2)$:
$$\exp(iL\delta t) \approx \underbrace{\exp(iL_4\delta t/4)\exp(iL_3\delta t/2)\exp(iL_4\delta t/4)}_{\text{Reservoir half-step}} \underbrace{\exp(iL_2\delta t/2)\exp(iL_1\delta t)\exp(iL_2\delta t/2)}_{\text{Velocity Verlet step}} \underbrace{\exp(iL_4\delta t/4)\exp(iL_3\delta t/2)\exp(iL_4\delta t/4)}_{\text{Reservoir half-step}}$$
Because the sequence of operators reads the same forwards and backwards, this propagator is **strictly time-reversible** and preserves the extended phase-space measure!

### The Harmonic Oscillator Non-Ergodicity Crisis & Nosé–Hoover Chains
As highlighted on Slides 11–13 (Frenkel & Smit Figures 6.7 and 6.8), a single Nosé–Hoover thermostat relies on the assumption that the extended system is **ergodic**—meaning there is only a single constant of motion ($H'$) and a single trajectory visits all states compatible with $H'$.

For a **1D harmonic oscillator** ($V(x) = \frac{1}{2}x^2, m=1$) or stiff harmonic phonon modes in a crystal, **a single Nosé–Hoover thermostat is NOT ergodic!** Instead of filling the 2D Gaussian phase-space cloud $(x, v)$, the trajectory remains trapped on a **hollow ring/torus** (Slide 12, right panel), never visiting the origin $(x=0, v=0)$!

**The Cure — Nosé–Hoover Chains (Martyna, Klein, Tuckerman 1992):** We thermostat the primary thermostat variable $\xi_1$ with a second thermostat variable $\xi_2$ (and generally a chain of $M$ bath variables $\xi_1, \dots, \xi_M$ with masses $Q_1, \dots, Q_M$). For a 2-link chain on a 1D harmonic oscillator (Slide 13):
$$\dot{r} = v, \qquad \dot{v} = -r - \xi_1 v, \qquad \dot{\xi}_1 = \frac{v^2 - T}{Q_1} - \xi_1\xi_2, \qquad \dot{\xi}_2 = \frac{Q_1\xi_1^2 - T}{Q_2}$$
Fluctuations of the second bath variable $\xi_2$ break the regular invariant tori of $\xi_1$, restoring full ergodicity and recovering the exact Gaussian distributions for both $P(x)$ and $P(v)$!


In [29]:
ws.render_section('concept_nh_ergodicity')

### Interactive Explorer: Harmonic Oscillator Phase Space — Single Nosé–Hoover vs. Andersen vs. Nosé–Hoover Chain

Switch between **Velocity Rescaling**, **Single Nosé–Hoover**, **Andersen Stochastic**, and **Nosé–Hoover Chain** for a 1D harmonic oscillator (reproducing Frenkel & Smit Figures 6.7 and 6.8 from Slides 12–13). Observe both the 2D phase-space portrait $(q, p)$ and the marginal position distribution $P(q)$ plotted along the bottom axis.


**Key Takeaway:** Even when a single Nosé–Hoover thermostat produces a reasonable-looking velocity distribution $P(v)$, its position distribution $P(q)$ for a harmonic mode is bimodal (hollow ring) rather than Gaussian! **Nosé–Hoover Chains** and the **Andersen thermostat** both fill the interior of phase space and reproduce the exact canonical Gaussian.


In [30]:
ws.render_section('q22_trotter_l3_momentum_action')

**Q22 · medium** · 💡

In the Martyna et al. (1996) Liouville operator-splitting propagator for the Nosé–Hoover thermostat (Slides 9–10), the friction sub-operator is $iL_3 = \zeta\frac{\partial}{\partial \eta} - \zeta\mathbf{p}\cdot\frac{\partial}{\partial \mathbf{p}}$. What is the exact analytical action of the finite-step propagator $\exp(iL_3\delta t)$ on the particle momentum $\mathbf{p}$ (treating $\zeta$ as constant during the sub-step)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Solve $\frac{d\mathbf{p}}{dt} = -\zeta \mathbf{p}$ over a time interval $\delta t$, or check Slide 10 directly.

</div>
</details>

- [ ] **(A)** $\exp(iL_3\delta t)\mathbf{p} = \mathbf{p}\exp(-\zeta\,\delta t)$
- [ ] **(B)** $\exp(iL_3\delta t)\mathbf{p} = \mathbf{p} - \zeta\,\delta t$
- [ ] **(C)** $\exp(iL_3\delta t)\mathbf{p} = \mathbf{p} / \zeta$
- [ ] **(D)** $\exp(iL_3\delta t)\mathbf{p} = \mathbf{p} + \mathbf{f}\,\delta t$

In [31]:
ws.render_section('q23_trotter_l3_numeric')

**Q23 · medium** · 💡

During a Nosé–Hoover operator-splitting half-step $\exp(iL_3 \delta t/2)\mathbf{p} = \mathbf{p}\exp(-\zeta \delta t/2)$, a particle enters the sub-step with momentum $p_x = 4.00$, the current friction coefficient is $\zeta = 0.50$, and the full MD time step is $\delta t = 0.40$ (so $\delta t/2 = 0.20$). What is the updated momentum $p_x'$ after applying $\exp(iL_3 \delta t/2)$ (to three decimal places)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $4.00 \times \exp(-0.50 \times 0.20) = 4.00 \times e^{-0.10}$.

</div>
</details>

_Numeric answer required._

In [32]:
ws.render_section('q24_trotter_symmetry_purpose')

**Q24 · medium** · 💡

Why does the Martyna–Tuckerman–Tobias–Klein (MTTK) propagator split the Liouville operator symmetrically as
$$\exp(iL\delta t) \approx \exp(i(L_3+L_4)\delta t/2)\,\exp(i(L_1+L_2)\delta t)\,\exp(i(L_3+L_4)\delta t/2)$$
rather than simply applying $\exp(iL_1\delta t)\exp(iL_2\delta t)\exp(iL_3\delta t)\exp(iL_4\delta t)$ sequentially in one direction?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What happens when you invert a palindromic product of operators $A(\delta t/2) B(\delta t) A(\delta t/2)$ by sending $\delta t \to -\delta t$?

</div>
</details>

- [ ] **(A)** Symmetric Trotter splitting guarantees strict time-reversibility ($U(-\delta t) = U(\delta t)^{-1}$) and cancels $\mathcal{O}(\delta t^2)$ local splitting errors to make the integrator second-order accurate.
- [ ] **(B)** Symmetric splitting eliminates the need to compute interatomic forces $\mathbf{F}(\mathbf{r})$.
- [ ] **(C)** Sequential splitting only works in 1D, whereas symmetric splitting works in 3D.
- [ ] **(D)** Symmetric splitting forces $\zeta(t)$ to remain strictly positive.

In [33]:
ws.render_section('q25_harmonic_oscillator_failure')

**Q25 · medium** · 💡

In Frenkel & Smit Figure 6.7 (Slide 12) and the interactive harmonic-oscillator explorer, what striking defect appears in the $(x, v)$ phase-space trajectory when a **single Nosé–Hoover thermostat** is applied to a 1D harmonic oscillator?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare the middle plot ('Andersen') with the right plot ('Nose') at the bottom of Slide 12.

</div>
</details>

- [ ] **(A)** The trajectory spirals outward to infinity because the extended energy $H'$ is not conserved.
- [ ] **(B)** The trajectory is non-ergodic and remains confined to a hollow ring/torus band, completely failing to sample the interior near $(x=0, v=0)$ required by a 2D Gaussian canonical distribution.
- [ ] **(C)** The velocity $v$ collapses to zero while position $x$ executes a random walk.
- [ ] **(D)** The trajectory only visits the first quadrant ($x > 0, v > 0$).

In [34]:
ws.render_section('q26_nh_chain_equations')

**Q26 · medium** · 💡

In a 2-link **Nosé–Hoover Chain** for a 1D harmonic oscillator at target temperature $T$ (Slide 13, $\dot{r}=v$, $\dot{v}=-r-\xi_1 v$, $\dot{\xi}_1 = \frac{v^2 - T}{Q_1} - \xi_1\xi_2$), what is the equation of motion for the **second** thermostat variable $\xi_2$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

The second thermostat $\xi_2$ regulates the 'kinetic energy' $Q_1 \xi_1^2$ of the first thermostat $\xi_1$ toward $T$.

</div>
</details>

- [ ] **(A)** $\dot{\xi}_2 = \frac{Q_1 \xi_1^2 - T}{Q_2}$
- [ ] **(B)** $\dot{\xi}_2 = \frac{v^2 - T}{Q_2}$
- [ ] **(C)** $\dot{\xi}_2 = \frac{r^2 - T}{Q_2}$
- [ ] **(D)** $\dot{\xi}_2 = -\xi_1 \xi_2$

In [35]:
ws.render_section('q27_nh_chain_drive_numeric')

**Q27 · medium** · 💡

In the 2-link Nosé–Hoover Chain equations on Slide 13,
$$\dot{\xi}_1 = \frac{v^2 - T}{Q_1} - \xi_1\xi_2, \qquad \dot{\xi}_2 = \frac{Q_1\xi_1^2 - T}{Q_2}$$
suppose $T = 1.0$, $Q_1 = 2.0$, $Q_2 = 4.0$, and at the current instant $v = 2.0$, $\xi_1 = 1.5$, and $\xi_2 = 0.5$. Compute the instantaneous value of $\dot{\xi}_2$.


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute $Q_1 = 2.0$, $\xi_1 = 1.5$, $T = 1.0$, and $Q_2 = 4.0$ into $\dot{\xi}_2 = (Q_1 \xi_1^2 - T) / Q_2$.

</div>
</details>

_Numeric answer required._

In [36]:
ws.render_section('q28_nh_subtle_issues')

**Q28 · medium** · 💡

According to Slide 11 ("How natural is the thermostat?"), what are the physical limitations of a standard global Nosé–Hoover thermostat even when it is time-reversible and deterministic? *(Select all that apply.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check Slide 11: how does a global friction coefficient $\zeta(t)$ interact with spatially separated atoms, and what happens with a single heat-bath variable?

</div>
</details>

- [ ] **(A)** The thermostat interactions are non-local and propagate instantaneously across the entire simulation box via the shared scalar variable $\zeta(t)$.
- [ ] **(B)** Coupling the entire system to a single heat-bath variable can produce strange non-ergodic dynamics (especially for harmonic or low-dimensional modes) if the mass $Q$ is not carefully chosen or chained.
- [ ] **(C)** It is needed in non-equilibrium steady-state simulations where external work creates a net flux of energy into or out of the system.
- [ ] **(D)** It destroys velocity continuity at every time step.

In [37]:
ws.render_section('sec_constant_pressure_and_pr')

### Constant Pressure ($P$-$T$) vs. Constant Volume ($V$-$T$) Phase Transitions
Why is constant-pressure ($NPT$) simulation vastly preferable to constant-volume ($NVT$) when studying first-order phase transitions (melting, vaporization, or solid-solid polymorphic transitions)?

As shown on Slide 14:
- **At Constant Pressure ($P$-$T$ diagram):** First-order phase transitions occur at a **sharp transition temperature $T_m$** where the volume $V(T)$ jumps discontinuously from $V_{\text{solid}}$ to $V_{\text{liquid}}$. The entire simulation box can convert cleanly from one homogeneous phase to the other.
- **At Constant Volume ($V$-$T$ diagram):** A single first-order transition line opens up into a broad **two-phase coexistence region** (shaded region on Slide 14). If the fixed density $N/V$ lies inside this region, the finite simulation cell must split into **coexisting droplets/crystallites separated by an interface**. Because creating a solid-liquid or liquid-vapor interface carries a large positive **surface tension (interfacial free energy) penalty** $\Delta F_{\text{surf}} \propto \gamma L^{d-1}$, nucleation is severely suppressed by finite-size barriers and hysteresis!
- **Box Shape Frustration:** Even if volume $V$ can change isotropically, a cubic periodic box strongly favors cubic crystal structures (e.g., FCC or rocksalt) and geometrically frustrates transitions to non-cubic or sheared unit cells!

### Virial Pressure and Extended-Lagrangian Barostats
Just as a thermostat couples particle momenta to a heat bath at temperature $T$ via kinetic temperature $T_I$, a **barostat** couples the simulation box dimensions to an external pressure bath $P_{\text{ext}}$ via the internal **Virial Pressure** (for box volume $\Omega = V$):
$$P_{\text{int}} = \frac{1}{3\Omega}\sum_{i=1}^N \left(\frac{p_i^2}{m_i} + \mathbf{r}_i \cdot \mathbf{F}_i\right)$$
and more generally the microscopic **$3\times 3$ internal stress tensor $\boldsymbol{\sigma}$** (Slide 17):
$$\sigma_{\alpha\beta} = \frac{1}{\Omega}\sum_{i=1}^N \left(r_{i\alpha} F_{i\beta} + m_i v_{i\alpha} v_{i\beta}\right)$$

1. **Andersen Isotropic Barostat (1980):** Treats the scalar volume $\Omega(t)$ as a dynamical piston with fictitious piston mass $W$, using scaled fractional coordinates $\mathbf{s}_i = \Omega^{-1/3}\mathbf{r}_i \in [0, 1)^3$. The box expands when $P_{\text{int}} > P_{\text{ext}}$ and contracts when $P_{\text{int}} < P_{\text{ext}}$, while keeping the box angles locked (e.g., a cube stays a cube).
2. **Parrinello–Rahman Anisotropic Variable-Cell Barostat (1981):** Generalizes the box into a **time-dependent $3\times 3$ cell matrix $\mathbf{L}(t) = [\mathbf{a}(t), \mathbf{b}(t), \mathbf{c}(t)]$** (with volume $\Omega = \det\mathbf{L}$):
   $$\mathbf{r}_i = \mathbf{L}\,\mathbf{s}_i, \qquad \mathbf{s}_i = \mathbf{L}^{-1}\mathbf{r}_i \quad \text{with internal fractional coordinates } 0 \le s_{i\alpha} < 1$$
   - **Why restrict $\mathbf{L}$ to be a symmetric $3\times 3$ matrix ($\mathbf{L} = \mathbf{L}^T$)?** A general $3\times 3$ matrix has 9 components, of which 3 are pure rigid-body rotations of the simulation box in space. Forcing $\mathbf{L}$ to be **symmetric** eliminates artificial cell rotation, leaving $6$ independent strain/shape degrees of freedom!
   - **How PBCs and forces are evaluated (Slide 16):** Apply periodic boundary conditions trivially in internal coordinates $s_{i\alpha} \leftarrow s_{i\alpha} - \text{nint}(s_{i\alpha})$ on the unit cube $[0,1)^3$, then transform to physical coordinates $\mathbf{r}_{ij} = \mathbf{L}\mathbf{s}_{ij}$ (or use the metric tensor $\mathbf{G} = \mathbf{L}^T\mathbf{L}$ via $r_{ij}^2 = \mathbf{s}_{ij}^T \mathbf{G} \mathbf{s}_{ij}$) to calculate physical energies and forces $\mathbf{F}_i(\mathbf{r})$.

### Parrinello–Rahman Equations of Motion & KCl B1 $\to$ B2 Transition
Assigning fictitious cell mass/response parameter $\omega$ (often denoted $W$) to the cell matrix $\mathbf{L}$, the equations of motion (Slide 17) are:
$$\frac{d\mathbf{p}_i}{dt} = \mathbf{F}_i - \mathbf{L}^{-T}\frac{d\mathbf{L}}{dt}\mathbf{p}_i, \qquad \omega \frac{d^2\mathbf{L}}{dt^2} = \left(\boldsymbol{\sigma} - P\mathbf{1}\right)\Omega\,\mathbf{L}^{-T}$$
generating the extended isobaric distribution $\propto \exp\!\left[-\beta\left(H + P\Omega + \text{kinetic cell term}\right)\right]$.

In their famous 1981 demonstration (Slide 18), Parrinello and Rahman simulated $N = 500$ $\text{KCl}$ ions at $T = 300\text{ K}$. Starting at $P = 0$ in the 6-coordinated **rocksalt (B1)** structure and compressing to $P = 44\text{ kbar}$, the cell spontaneously sheared and dilated along a body diagonal into the denser 8-coordinated **cesium chloride (B2)** structure within $\sim 0.3\text{ ps}$, and reversed back to rocksalt upon decompression!

> [!IMPORTANT] Caveats of Variable-Cell Dynamics (Slide 19)
> - **Unrealistic cell dynamics:** Because the cell walls move globally and instantaneously in response to $\boldsymbol{\sigma}$ ($\mathcal{O}(1/N)$ non-local coupling), the kinetics of box-shape fluctuations are artificial—just because a pathway exists does not mean its transition barrier or rate is physical.
- **Metastable trapping:** Variable-cell MD is **not** guaranteed to escape local free-energy minima; rigorous **free-energy calculations** ($G = E + PV - TS$) are required to establish which crystal polymorph is truly thermodynamically stable.


In [38]:
ws.render_section('concept_parrinello_rahman')

### Interactive Explorer: Parrinello–Rahman Variable-Shape Cell vs. Fixed-Shape Frustration (KCl B1 → B2)

Explore how internal fractional coordinates $\mathbf{s}_i \in (0, 1)^2$ map to physical coordinates $\mathbf{r}_i = \mathbf{L}\mathbf{s}_i$ during a high-pressure solid-solid transformation (inspired by the $44\text{ kbar}$ $\text{KCl}$ rocksalt B1 $\to$ CsCl B2 transition on Slide 18). Compare **Parrinello–Rahman** variable-shape relaxation against **Andersen Isotropic** and **Fixed Box (NVT)** modes.


**Key Takeaway:** In internal coordinates $\mathbf{s}_i = \mathbf{L}^{-1}\mathbf{r}_i$, periodic boundary conditions always live on the simple unit cube $0 \le s_{i\alpha} < 1$. When $\mathbf{L}(t)$ is allowed to shear symmetrically in response to anisotropic stress $(\boldsymbol{\sigma} - P\mathbf{1})$, the crystal transforms homogeneously without forming artificial domain walls!


In [39]:
ws.render_section('q29_constant_v_vs_p_coexistence')

**Q29 · easy** · 💡

Why is a first-order solid–liquid melting transition much sharper and easier to observe in a **constant-pressure ($NPT$)** simulation than in a **constant-volume ($NVT$)** simulation of a finite periodic cell (Slide 14)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the $V$-$T$ vs. $P$-$T$ diagrams on Slide 14 and consider what happens inside the shaded two-phase region at fixed volume.

</div>
</details>

- [ ] **(A)** At constant volume, the system is forced into a two-phase coexistence region where forming solid/liquid droplets requires overcoming a large surface-tension (interfacial free energy) barrier, whereas at constant pressure the box volume jumps cleanly across a sharp transition line.
- [ ] **(B)** Constant-volume simulations cannot compute interatomic forces when a liquid forms.
- [ ] **(C)** Temperature cannot be controlled in a constant-volume simulation.
- [ ] **(D)** At constant pressure, surface tension becomes negative so droplets explode.

In [40]:
ws.render_section('q30_virial_pressure_numeric')

**Q30 · medium** · 💡

Using the microscopic Virial Pressure formula on Slide 15,
$$P = \frac{1}{3\Omega}\sum_{i=1}^N \left(\frac{p_i^2}{m_i} + \mathbf{r}_i \cdot \mathbf{F}_i\right)$$
consider a simulation cell of volume $\Omega = 200.0$ where the total kinetic term is $\sum_{i=1}^N \frac{p_i^2}{m_i} = 2K = 900.0$ and the internal virial is $\sum_{i=1}^N \mathbf{r}_i \cdot \mathbf{F}_i = 300.0$. What is the instantaneous internal pressure $P$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Add the kinetic and virial sums ($900.0 + 300.0 = 1200.0$) and divide by $3\Omega = 600.0$.

</div>
</details>

_Numeric answer required._

In [41]:
ws.render_section('q31_pbc_fractional_coords')

**Q31 · easy** · 💡

In a Parrinello–Rahman variable-cell simulation where physical coordinates $\mathbf{r}_i$ and internal fractional coordinates $\mathbf{s}_i \in [0, 1)^3$ are related by $\mathbf{r}_i = \mathbf{L}\mathbf{s}_i$ (Slide 16), which coordinates are used to apply **periodic boundary conditions (minimum-image wrapping)** and which are used to evaluate **interatomic potential energies and forces**?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the last two bullet points on Slide 16!

</div>
</details>

- [ ] **(A)** Periodic boundary conditions are applied in internal coordinates $\mathbf{s}_i$ (unit cube $[0,1)^3$), while energies and forces are calculated using physical coordinates $\mathbf{r}_i = \mathbf{L}\mathbf{s}_i$.
- [ ] **(B)** Both periodic boundary conditions and interatomic forces are evaluated directly in $\mathbf{s}_i$ without ever using $\mathbf{L}$.
- [ ] **(C)** Periodic boundary conditions are applied in physical coordinates $\mathbf{r}_i$ assuming a unit cube, while forces are evaluated in $\mathbf{s}_i$.
- [ ] **(D)** Periodic boundary conditions are disabled in Parrinello–Rahman MD.

In [42]:
ws.render_section('q32_symmetric_L_matrix_reason')

**Q32 · medium** · 💡

Why is the $3\times 3$ cell matrix $\mathbf{L}(t)$ in the Parrinello–Rahman barostat constrained to be **symmetric** ($\mathbf{L} = \mathbf{L}^T$, reducing its 9 components to 6 independent variables), as noted on Slide 16?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

A general $3\times 3$ deformation gradient $\mathbf{L}$ can be polar-decomposed into a symmetric stretch tensor and an orthogonal rotation matrix. What do the 3 antisymmetric degrees of freedom represent?

</div>
</details>

- [ ] **(A)** To eliminate overall rigid-body rotation (angular momentum) of the simulation cell in space.
- [ ] **(B)** Because only diagonal matrices are symmetric, which forces the cell to remain cubic.
- [ ] **(C)** Because the determinant of a non-symmetric matrix is always zero.
- [ ] **(D)** To force the internal virial pressure to be zero.

In [43]:
ws.render_section('q33_coordinate_mapping_numeric')

**Q33 · medium** · 💡

Consider a 2D sheared Parrinello–Rahman cell with symmetric transformation matrix
$$\mathbf{L} = \begin{pmatrix} 10.0 & 2.0 \\ 2.0 & 10.0 \end{pmatrix}\text{ Å}$$
An atom is located at internal fractional coordinates $\mathbf{s} = (0.50, 0.25)^T$. Using $\mathbf{r} = \mathbf{L}\mathbf{s}$, what is its physical $x$-coordinate $r_x$ (in $\text{Å}$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute the first row of $\mathbf{L}\mathbf{s}$: $r_x = L_{xx} s_x + L_{xy} s_y = 10.0(0.50) + 2.0(0.25)$.

</div>
</details>

_Numeric answer required._

In [44]:
ws.render_section('q34_cell_volume_det_numeric')

**Q34 · medium** · 💡

A 3D orthorhombic simulation cell has diagonal cell matrix $\mathbf{L} = \text{diag}(10.0, 10.0, 10.0)\text{ Å}$ at $P = 0$ (volume $\Omega_0 = 1000\text{ Å}^3$). Under shear and compression, the symmetric cell matrix evolves to
$$\mathbf{L} = \begin{pmatrix} 9.0 & 2.0 & 0.0 \\ 2.0 & 9.0 & 0.0 \\ 0.0 & 0.0 & 10.0 \end{pmatrix}\text{ Å}$$
What is the new cell volume $\Omega = \det(\mathbf{L})$ (in $\text{Å}^3$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

For this block-diagonal matrix, $\det(\mathbf{L}) = 10.0 \times (9.0 \times 9.0 - 2.0 \times 2.0)$.

</div>
</details>

_Numeric answer required._

In [45]:
ws.render_section('q35_pr_cell_acceleration_equation')

**Q35 · medium** · 💡

On Slide 17, the Parrinello–Rahman equation of motion for the cell matrix $\mathbf{L}(t)$ with cell inertia/response parameter $\omega$ and external hydrostatic pressure $P$ is written in terms of the microscopic internal stress tensor $\boldsymbol{\sigma}$ as:


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the second equation on Slide 17: what tensor difference drives $\omega \frac{d^2\mathbf{L}}{dt^2}$?

</div>
</details>

- [ ] **(A)** $\omega \frac{d^2\mathbf{L}}{dt^2} = (\boldsymbol{\sigma} - P\mathbf{1})\Omega\mathbf{L}^{-T}$
- [ ] **(B)** $\omega \frac{d^2\mathbf{L}}{dt^2} = -P\mathbf{L}$
- [ ] **(C)** $\omega \frac{d\mathbf{L}}{dt} = \text{Tr}(\boldsymbol{\sigma})\mathbf{1}$
- [ ] **(D)** $\omega \frac{d^2\mathbf{L}}{dt^2} = \sum_i \mathbf{F}_i$

In [46]:
ws.render_section('q36_kcl_b1_to_b2_transition')

**Q36 · easy** · 💡

In the classic Parrinello–Rahman (1981) variable-cell MD simulation of $500$ $\text{KCl}$ ions at $300\text{ K}$ (Slide 18), what structural transformation occurs spontaneously when the external pressure is stepped from $P = 0$ to $P = 44\text{ kbar}$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the bullet points and Figure 3 caption on Slide 18.

</div>
</details>

- [ ] **(A)** The crystal transforms from the 6-coordinated rocksalt (NaCl / B1) structure to the denser 8-coordinated cesium chloride (CsCl / B2) structure via a body-diagonal dilatation and shear of the unit cell.
- [ ] **(B)** The KCl crystal sublimates into a diatomic ideal gas.
- [ ] **(C)** The KCl ions disproportionate into metallic potassium and $\text{Cl}_2$ gas.
- [ ] **(D)** The volume increases by $50\%$ while remaining in the rocksalt structure.

In [47]:
ws.render_section('q37_kcl_temp_spike_reason')

**Q37 · hard** · 💡

Inspect the top panel of Figure 2 on Slide 18 (the time history of temperature $T(t)$ during the $\text{KCl}$ compression run). When the system transitions from metastable rocksalt (region II) into the stable $\text{CsCl}$ structure (region III, arrows #2 and #3), the temperature $T$ suddenly jumps from $\sim 700\text{ K}$ to $\sim 1250\text{ K}$ at the exact same moment the density jumps upward. What causes this sudden temperature spike?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

When a metastable high-enthalpy phase drops into a lower-enthalpy stable crystal structure at constant enthalpy ($NPH$), where does the latent enthalpy $\Delta H = \Delta E + P\Delta V$ go?

</div>
</details>

- [ ] **(A)** Release of latent heat (enthalpy of transformation $\Delta H = \Delta V_{\text{pot}} + P\Delta \Omega < 0$): as the ions drop into the lower-enthalpy CsCl structure and $P\Delta \Omega$ work is done on the compressing cell, potential enthalpy converts into kinetic energy.
- [ ] **(B)** A bug in the periodic boundary conditions that duplicates atoms across the cell wall.
- [ ] **(C)** The Boltzmann constant $k_B$ changes value inside the CsCl phase.
- [ ] **(D)** The barostat mass $\omega$ becomes negative in region III.

In [48]:
ws.render_section('q38_metric_tensor_distance')

**Q38 · hard** · 💡

Given the Parrinello–Rahman coordinate mapping $\mathbf{r}_i = \mathbf{L}\mathbf{s}_i$, how is the squared physical distance $r_{ij}^2 = |\mathbf{r}_i - \mathbf{r}_j|^2$ between two atoms expressed directly in terms of their internal fractional separation vector $\mathbf{s}_{ij} = \mathbf{s}_i - \mathbf{s}_j$ and the metric tensor $\mathbf{G} = \mathbf{L}^T\mathbf{L}$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Write $r_{ij}^2 = (\mathbf{L}\mathbf{s}_{ij})^T (\mathbf{L}\mathbf{s}_{ij})$ and use the transpose product identity $(\mathbf{A}\mathbf{B})^T = \mathbf{B}^T \mathbf{A}^T$.

</div>
</details>

- [ ] **(A)** $r_{ij}^2 = \mathbf{s}_{ij}^T \mathbf{L}^T \mathbf{L}\,\mathbf{s}_{ij} = \mathbf{s}_{ij}^T \mathbf{G}\,\mathbf{s}_{ij}$
- [ ] **(B)** $r_{ij}^2 = \det(\mathbf{L})\,|\mathbf{s}_{ij}|^2$
- [ ] **(C)** $r_{ij}^2 = \text{Tr}(\mathbf{L})\,(\mathbf{s}_i \cdot \mathbf{s}_j)$
- [ ] **(D)** $r_{ij}^2 = \mathbf{s}_{ij}^T \mathbf{L}^{-1} \mathbf{s}_{ij}$

In [49]:
ws.render_section('q39_variable_cell_caveats_multi')

**Q39 · medium** · 💡

According to Slide 19 ("Constant Pressure / Variable Cell Simulations"), which of the following cautions must a researcher keep in mind when using Parrinello–Rahman variable-cell MD to search for crystal structures? *(Select all that apply.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review the five bullet points on Slide 19 regarding local minima, free energy calculations, and non-locality.

</div>
</details>

- [ ] **(A)** Variable-cell MD is not guaranteed to escape a local free-energy minimum within a finite simulation run, so one can get trapped in a metastable polymorph and obtain the wrong structure.
- [ ] **(B)** Careful free-energy calculations ($G = E + PV - TS$) are needed to rigorously establish which crystal structure is thermodynamically stable.
- [ ] **(C)** Variable-cell methods have non-physical dynamics because the global boundary coupling does not respect spatial locality of interactions (though equilibrium thermodynamic finite-size effects are $\mathcal{O}(1/N)$).
- [ ] **(D)** Parrinello–Rahman MD is mathematically guaranteed to find the global ground-state crystal structure within 100 time steps for any material.

In [50]:
ws.render_section('q40_andersen_vs_pr_dof')

**Q40 · easy** · 💡

How many independent extended **box degrees of freedom** are added to the Lagrangian in the **Andersen isotropic barostat** versus the **symmetric Parrinello–Rahman anisotropic barostat** in 3D?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

An isotropic barostat varies only the scalar volume $\Omega$, whereas a symmetric $3\times 3$ matrix $\mathbf{L} = \mathbf{L}^T$ has how many independent entries?

</div>
</details>

- [ ] **(A)** 1 scalar volume degree of freedom ($\Omega$) in Andersen's isotropic barostat vs. 6 independent cell-matrix degrees of freedom ($L_{\alpha\beta} = L_{\beta\alpha}$) in Parrinello–Rahman.
- [ ] **(B)** 3 degrees of freedom in Andersen vs. 27 degrees of freedom in Parrinello–Rahman.
- [ ] **(C)** 6 degrees of freedom in both barostats.
- [ ] **(D)** 1 degree of freedom in both barostats.

In [51]:
ws.render_section('q41_comprehensive_thermostat_comparison')

**Q41 · medium** · 💡

You want to compute the **canonical ($NVT$) equilibrium heat capacity** $C_V = \frac{\langle \Delta E^2 \rangle_{NVT}}{k_B T^2}$ and the **vibrational density of states** (from continuous velocity time-correlation functions) of a harmonic crystal with $N = 500$ atoms. Which thermostat choice is **most appropriate** for simultaneously achieving ergodic canonical sampling and continuous, deterministic, time-reversible trajectories?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Eliminate Quench/Berendsen (wrong fluctuations), Andersen (discontinuous velocities that damp VACFs), and single Nosé–Hoover (non-ergodic for harmonic modes).

</div>
</details>

- [ ] **(A)** Velocity rescaling (Quench) applied at every time step.
- [ ] **(B)** Andersen thermostat with a very high collision rate $\nu \Delta t = 0.5$.
- [ ] **(C)** A single Nosé–Hoover thermostat with $Q = 10^8$.
- [ ] **(D)** A Nosé–Hoover Chain thermostat integrated with a symmetric Trotter (MTTK) splitting scheme and a well-matched thermal inertia $Q$.

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()